# DCDP linear inverse problems: final outputs only

This notebook runs the linear DCDP inverse-problem suite and saves only final outputs: final generated images, `generated_images.zip`, per-image final metrics, and progress/final run metadata. It does not enable checkpoint/anytime metric history, detailed per-image quality history, reconstruction trajectory tensors, or progress figures.

Included tasks: `super_resolution`, `inpainting_box`, `inpainting_random`, `gaussian_blur`, and `motion_blur`. Phase retrieval is intentionally excluded because it is nonlinear and uses a much heavier repo preset.

In [ ]:
# @title 1. Clone the DCDP repo and install dependencies
REPO_URL = "https://github.com/Seif-Hussein/Decoupled-Data-Consistency-with-Diffusion-Purification-for-Image-Restoration.git"  # @param {type:"string"}
BRANCH = "codex-dcdp-colab-defaults"  # @param {type:"string"}
REPO_DIR = "/content/dcdp"  # @param {type:"string"}
INSTALL_DEPS = True  # @param {type:"boolean"}

import os
import subprocess
import sys
from pathlib import Path

def run(cmd, cwd=None):
    print('+', ' '.join(str(c) for c in cmd))
    process = subprocess.Popen(
        [str(c) for c in cmd],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    for line in process.stdout:
        print(line, end='')
        tail.append(line)
        tail = tail[-300:]
    returncode = process.wait()
    if returncode:
        failure_log = Path('last_subprocess_failure.log')
        failure_log.write_text(''.join(tail))
        print(f'Command failed with exit code {returncode}. Tail saved to {failure_log.resolve()}')
        raise subprocess.CalledProcessError(returncode, cmd, output=''.join(tail))

repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, str(repo_dir)])
else:
    run(['git', 'fetch', 'origin', BRANCH], cwd=repo_dir)
    run(['git', 'checkout', BRANCH], cwd=repo_dir)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=repo_dir)
os.chdir(repo_dir)
run(['git', 'rev-parse', '--short', 'HEAD'])
print('Working directory:', Path.cwd())

if INSTALL_DEPS:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'PyYAML', 'matplotlib', 'scipy', 'tqdm', 'scikit-image', 'torchmetrics[image]', 'lpips'])

if not Path('motionblur').exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/LeviBorodenko/motionblur', 'motionblur'])

required = [
    Path('scripts/run_dcdp_default_inverse_pipeline.py'),
    Path('task_configurations/dcdp_user_inpainting_box_config.yaml'),
    Path('purification_configurations/dcdp_paper_motion_deblur.yaml'),
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise RuntimeError('Missing required Colab files: ' + ', '.join(missing))

print('Setup ready.')

In [ ]:
# @title 2. Prepare checkpoint and Drive data
MOUNT_DRIVE = True  # @param {type:"boolean"}
FFHQ_CHECKPOINT_FROM_DRIVE = "/content/drive/MyDrive/ffhq_10m.pt"  # @param {type:"string"}
DOWNLOAD_CHECKPOINT_IF_MISSING = True  # @param {type:"boolean"}
DATASET_ROOT = "/content/drive/MyDrive/mycode/test-ffhq"  # @param {type:"string"}

import gc
import hashlib
import shutil
import torch

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

Path('models').mkdir(exist_ok=True)
target_ckpt = Path('models/ffhq_10m.pt')
drive_ckpt = Path(FFHQ_CHECKPOINT_FROM_DRIVE)
if not target_ckpt.exists() and drive_ckpt.exists():
    shutil.copy2(drive_ckpt, target_ckpt)
    print('Copied checkpoint to', target_ckpt)
elif target_ckpt.exists():
    print('Checkpoint found:', target_ckpt)
elif DOWNLOAD_CHECKPOINT_IF_MISSING:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    run(['gdown', '--id', '1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh', '-O', str(target_ckpt)])
else:
    raise FileNotFoundError('Checkpoint not found. Put ffhq_10m.pt at models/ffhq_10m.pt or set FFHQ_CHECKPOINT_FROM_DRIVE.')

if not target_ckpt.exists():
    raise FileNotFoundError(f'Checkpoint missing after setup: {target_ckpt}')
ckpt_size = target_ckpt.stat().st_size
if ckpt_size < 100_000_000:
    raise RuntimeError(f'Checkpoint file is suspiciously small ({ckpt_size} bytes): {target_ckpt}')
hasher = hashlib.sha256()
with target_ckpt.open('rb') as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b''):
        hasher.update(chunk)
try:
    state_dict = torch.load(target_ckpt, map_location='cpu')
    if not isinstance(state_dict, dict) or not state_dict:
        raise RuntimeError('Checkpoint did not load as a non-empty state_dict.')
finally:
    try:
        del state_dict
    except NameError:
        pass
    gc.collect()
print(f'Checkpoint verified: {target_ckpt} ({ckpt_size / 1024**2:.1f} MB, sha256={hasher.hexdigest()[:12]}...)')

data_root = Path(DATASET_ROOT)
if not data_root.exists():
    raise FileNotFoundError(f'DATASET_ROOT does not exist: {data_root}')
print('Images found:', len(list(data_root.rglob('*.png'))), 'in', data_root)

In [ ]:
# @title 3. Run all linear tasks, final outputs only
TASKS = "super_resolution,inpainting_box,inpainting_random,gaussian_blur,motion_blur"  # @param {type:"string"}
MAX_IMAGES = 100  # @param {type:"integer"}
BATCH_SIZE = 10  # @param {type:"integer"}
DATA_START_IDX = 0  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
GPU = 0  # @param {type:"integer"}
SAVE_DIR = "purification_results/dcdp_linear_final_colab"  # @param {type:"string"}
MODE = "ddim"  # @param ["ddim", "tweedie"]
DRY_RUN = False  # @param {type:"boolean"}

cmd = [
    sys.executable,
    'scripts/run_dcdp_default_inverse_pipeline.py',
    '--tasks', TASKS,
    '--gpu', str(GPU),
    '--seed', str(SEED),
    '--max-images', str(MAX_IMAGES),
    '--start-idx', str(DATA_START_IDX),
    '--batch-size', str(BATCH_SIZE),
    '--dataset-root', DATASET_ROOT,
    '--save-dir', SAVE_DIR,
    '--mode', MODE,
    '--skip-metrics',
    '--final-metrics',
]
if DRY_RUN:
    cmd.append('--dry-run')
run(cmd)

# The core runner initializes empty checkpoint-history files. Remove them here
# so this notebook leaves only final-output artifacts.
if not DRY_RUN:
    import json
    for history_file in Path(SAVE_DIR).glob('*/metric_history.json'):
        payload = json.loads(history_file.read_text())
        if not payload.get('step') and not payload.get('elapsed_seconds_per_image'):
            history_file.unlink()
    for history_file in Path(SAVE_DIR).glob('*/quality_history.json'):
        payload = json.loads(history_file.read_text())
        if not payload.get('images'):
            history_file.unlink()

In [ ]:
# @title 4. Summarize final outputs
import json

def fmt_seconds(value):
    return 'n/a' if value is None else f'{value:.1f}s'

root = Path(SAVE_DIR)
progress_files = sorted(root.glob('*/progress.json'))
if not progress_files:
    print('No progress.json files found yet under', root)

for progress_path in progress_files:
    progress = json.loads(progress_path.read_text())
    run_info = progress.get('run', {})
    history_path = Path(progress.get('history_json') or progress_path.with_name('history.json'))
    images_zip = Path(progress.get('generated_images_zip') or run_info.get('generated_images_zip') or progress_path.parent / 'generated_images.zip')
    print('\nTask:', run_info.get('task_name', progress_path.parent.name))
    print('Status:', progress.get('status'))
    print('Completed:', progress.get('completed_images'), '/', run_info.get('target_images'))
    print('Batch size:', run_info.get('batch_size'))
    print('Average elapsed/image:', fmt_seconds(progress.get('avg_elapsed_seconds_per_image')))
    print('ETA:', fmt_seconds(progress.get('eta_seconds')))
    print('History/final ledger JSON:', history_path)
    print('Generated images zip:', images_zip, 'exists=' + str(images_zip.exists()))
    if history_path.exists():
        history = json.loads(history_path.read_text())
        images = history.get('images', [])
        metric_rows = [img.get('metrics', {}) for img in images if img.get('metrics')]
        if metric_rows:
            for key in ['final_psnr', 'final_ssim', 'final_lpips']:
                vals = [row[key] for row in metric_rows if key in row]
                if vals:
                    print(f'Average {key}:', sum(vals) / len(vals))
        if images:
            last = images[-1]
            print('Last output:', last.get('output_dir'))
            print('Last generated image:', last.get('generated_image'))
            print('Last final metrics:', last.get('metrics'))